# 01 — Reserve Training Node

**Purpose**: Reserve a Chameleon MI100 bare-metal node, assign a floating IP, and write the Ansible inventory.

**Prereqs**:
- `.env` populated (copy `.env.example`, fill in Chameleon credentials)
- SSH key pair registered with Nova at CHI@TACC

**Outcome**: `ansible/inventory.ini` written with the node's floating IP. Run `just provision` or proceed to `02_lerobot.ipynb`.

---

In [ ]:
import os
import socket
import time
import json
from datetime import datetime, timedelta
from pathlib import Path

import chi
from chi import lease, server, hardware
from chi.lease import Lease
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path('..') / '.env', override=False)

# Benchmark: record start time
_bench = {
    "notebook": "01_reserve_node",
    "started_at": datetime.utcnow().isoformat(),
    "timings": {}
}
_t0 = time.monotonic()

LEASE_NAME  = os.getenv("LEASE_NAME", "coachable-robots-mi100")
SERVER_NAME = os.getenv("SERVER_NAME", "coachable-training-node")
KEY_NAME    = os.getenv("KEY_PAIR_NAME")
LEASE_HOURS = int(os.getenv("LEASE_DURATION_HOURS", 6))
NODE_TYPE   = "gpu_mi100"
IMAGE_NAME  = "CC-Ubuntu22.04"

chi.use_site("CHI@TACC")
chi.set("project_name", os.getenv("OS_PROJECT_NAME"))

print(f"Site:       CHI@TACC")
print(f"Project:    {os.getenv('OS_PROJECT_NAME')}")
print(f"Lease:      {LEASE_NAME} ({LEASE_HOURS}h)")
print(f"Key pair:   {KEY_NAME}")

## 1. Check Existing Resources

In [ ]:
my_lease = None

# ── Existing leases ──
print("=" * 60)
print("EXISTING LEASES")
print("=" * 60)

all_leases = lease.list_leases()
active_leases = [l for l in all_leases if l.status in ("ACTIVE", "PENDING")]

if active_leases:
    for l in active_leases:
        marker = " <<<" if l.name == LEASE_NAME else ""
        print(f"  [{l.status}] {l.name}  (id: {l.id}){marker}")
        print(f"              ends: {l.end_date}")
        if l.node_reservations:
            print(f"              nodes: {len(l.node_reservations)} reservation(s)")
        if l.fip_reservations:
            print(f"              fips:  {len(l.fip_reservations)} reservation(s)")
        print()
        if l.name == LEASE_NAME:
            my_lease = l
else:
    print("  No active or pending leases.")

# ── Existing servers ──
print("=" * 60)
print("EXISTING SERVERS")
print("=" * 60)

try:
    existing_servers = server.list_servers()
    if existing_servers:
        for s in existing_servers:
            s_name = s.name if hasattr(s, 'name') else s.get('name', 'unknown')
            s_status = s.status if hasattr(s, 'status') else s.get('status', 'unknown')
            marker = " <<<" if s_name == SERVER_NAME else ""
            print(f"  [{s_status}] {s_name}{marker}")
    else:
        print("  No servers running.")
except Exception as e:
    print(f"  Could not list servers: {e}")

# ── MI100 availability ──
print()
print("=" * 60)
print("MI100 AVAILABILITY")
print("=" * 60)
available_nodes = hardware.get_nodes(node_type=NODE_TYPE, filter_reserved=True)
print(f"  {len(available_nodes)} '{NODE_TYPE}' node(s) available.")

print()
if my_lease:
    print(f"REUSING lease '{my_lease.name}' [{my_lease.status}]")
else:
    print(f"No active lease named '{LEASE_NAME}' — proceed to next cell to create one.")

## 2. Create Lease (if needed)

In [ ]:
_t_lease = time.monotonic()

if my_lease is None:
    if not available_nodes:
        print(f"WARNING: No {NODE_TYPE} nodes available. Check the host calendar.")
    else:
        resp = input(f"Create a {LEASE_HOURS}h lease for 1x {NODE_TYPE}? [y/N]: ")
        if resp.strip().lower() in ('y', 'yes'):
            my_lease = Lease(
                name=LEASE_NAME,
                duration=timedelta(hours=LEASE_HOURS),
            )
            my_lease.add_node_reservation(node_type=NODE_TYPE, amount=1)
            my_lease.add_fip_reservation(amount=1)
            my_lease.submit(
                wait_for_active=True,
                wait_timeout=600,
                show="widget",
                idempotent=True,
            )
            print(f"Lease ACTIVE: {my_lease.id}")
        else:
            print("Skipped. Re-run when ready.")
else:
    print(f"Using existing lease: {my_lease.id}")

_bench["timings"]["lease_s"] = round(time.monotonic() - _t_lease, 1)

## 3. Launch Server (Idempotent)

In [ ]:
if my_lease is None:
    raise RuntimeError("No lease. Run cell above and create one first.")

_t_server = time.monotonic()
gpu_server = None
floating_ip = None

# Check if server already exists
try:
    existing_id = server.get_server_id(SERVER_NAME)
    gpu_server_obj = server.get_server(existing_id)
    status = gpu_server_obj.status if hasattr(gpu_server_obj, 'status') else gpu_server_obj.get('status')

    if status == "ACTIVE":
        print(f"Server '{SERVER_NAME}' already ACTIVE — reusing.")
        gpu_server = gpu_server_obj
    elif status == "BUILD":
        print(f"Server still building — waiting...")
        server.wait_for_active(existing_id)
        gpu_server = server.get_server(existing_id)
    else:
        print(f"Server status is {status} — deleting and recreating.")
        server.delete_server(existing_id)
        time.sleep(10)
except Exception:
    print(f"No existing server '{SERVER_NAME}' — creating.")

if gpu_server is None:
    reservation_id = my_lease.node_reservations[0]["id"]
    print(f"Creating server from reservation {reservation_id}...")
    gpu_server = server.create_server(
        SERVER_NAME,
        reservation_id=reservation_id,
        image_name=IMAGE_NAME,
        key_name=KEY_NAME,
    )
    server.wait_for_active(gpu_server.id)
    print("Server ACTIVE.")

# Attach or find floating IP
try:
    ips = server.list_floating_ips(gpu_server.id) if hasattr(server, 'list_floating_ips') else []
    if ips:
        floating_ip = ips[0]
        print(f"Existing floating IP: {floating_ip}")
except Exception:
    pass

if not floating_ip:
    floating_ip = server.associate_floating_ip(gpu_server.id)
    print(f"Assigned floating IP: {floating_ip}")

print(f"\nSSH: ssh cc@{floating_ip}")
_bench["timings"]["server_boot_s"] = round(time.monotonic() - _t_server, 1)

## 4. Wait for SSH

In [ ]:
from tqdm.notebook import tqdm

print(f"Waiting for SSH on {floating_ip}...")
print("(Bare metal takes 5–15 min to image + boot on first launch.)")

_t_ssh = time.monotonic()
timeout = 900  # 15 min
interval = 15

with tqdm(total=timeout, unit="s", desc="SSH wait", bar_format="{desc}: {elapsed}") as pbar:
    elapsed = 0
    while elapsed < timeout:
        try:
            with socket.create_connection((floating_ip, 22), timeout=10):
                print(f"\nSSH ready after {elapsed:.0f}s")
                break
        except OSError:
            time.sleep(interval)
            elapsed += interval
            pbar.update(interval)
    else:
        print(f"\nTimed out after {timeout}s — check Chameleon dashboard.")

_bench["timings"]["ssh_wait_s"] = round(time.monotonic() - _t_ssh, 1)

## 5. Verify MI100 GPU

In [ ]:
from chi import ssh

# MI100 is AMD — use lspci and rocm-smi, NOT nvidia-smi
with ssh.Remote(floating_ip) as conn:
    print("=== OS ===")
    conn.run("lsb_release -ds")

    print("\n=== AMD GPU ===")
    conn.run("lspci | grep -i 'display\|vga\|amd\|radeon\|arcturus'")

    print("\n=== Kernel ===")
    conn.run("uname -r")

    print("\n=== Memory ===")
    conn.run("free -h | head -2")

## 6. Write Ansible Inventory

In [ ]:
PRIVATE_KEY = os.path.expanduser("~/.ssh/id_rsa")

inventory_path = Path("..") / "ansible" / "inventory.ini"
inventory_content = f"""[training]
mi100 ansible_host={floating_ip} ansible_user=cc ansible_ssh_private_key_file={PRIVATE_KEY}

[training:vars]
ansible_ssh_common_args=-o StrictHostKeyChecking=no
"""
inventory_path.write_text(inventory_content)

print(f"Inventory written: {inventory_path}")
print(f"  Target: cc@{floating_ip}")
print(f"  Key:    {PRIVATE_KEY}")
print()
print("Next steps:")
print("  just provision        → run Ansible playbook")
print("  just ssh-node         → SSH to the node")
print("  02_lerobot.ipynb      → collect + train workflow")

## Benchmark: Save Timing

In [ ]:
_bench["timings"]["total_s"] = round(time.monotonic() - _t0, 1)
_bench["completed_at"] = datetime.utcnow().isoformat()
_bench["floating_ip"] = floating_ip
_bench["lease_id"] = my_lease.id if my_lease else None

results_dir = Path("..") / "bench" / "results"
results_dir.mkdir(exist_ok=True)
ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
bench_path = results_dir / f"01_reserve_node_{ts}.json"
bench_path.write_text(json.dumps(_bench, indent=2))
print(json.dumps(_bench, indent=2))
print(f"\nBenchmark saved: {bench_path}")